# Reconocimiento de Lengua de Señas Peruana (LSP) - Robot NAO
## Práctica N.° 2 - Inteligencia Artificial (UPC)

Este cuaderno implementa el pipeline completo de entrenamiento, evaluación comparativa entre **Support Vector Machine (RBF)** y **Random Forest Classifier**, y la serialización del modelo ganador para su integración con el robot humanoide NAO.

In [ ]:
# 1. Configuración de dependencias e importación de librerías
!pip install --quiet scikit-learn seaborn matplotlib pandas numpy joblib

import os
import io
import time
from typing import Any
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

%matplotlib inline
print('Entorno inicializado con éxito.')

### 2. Carga del Dataset Consolidado (12,046 muestras)
Si ejecutas este cuaderno en Google Colab, se te solicitará subir el archivo `dataset_lsp_consolidado.csv`.

In [ ]:
from google.colab import files

RUTA_CSV = 'dataset_lsp_consolidado.csv'

if not os.path.exists(RUTA_CSV):
    print('Por favor, sube tu archivo dataset_lsp_consolidado.csv:')
    subidos = files.upload()
    nombre_subido = list(subidos.keys())[0]
    if nombre_subido != RUTA_CSV:
        os.rename(nombre_subido, RUTA_CSV)

print(f'Dataset disponible en: {RUTA_CSV}')

### 3. Validación de Dimensiones, Balanceo y Partición Estratificada (80/20)

In [ ]:
df = pd.read_csv(RUTA_CSV)
if df.isnull().sum().sum() > 0:
    df = df.dropna()

COLUMNAS_METADATOS = ['label', 'class_name', 'integrante']
columnas_caracteristicas = [c for c in df.columns if c not in COLUMNAS_METADATOS]

df_clases = df[['label', 'class_name']].drop_duplicates().sort_values('label')
nombres_clases = df_clases['class_name'].tolist()

matriz_x = df[columnas_caracteristicas].to_numpy(dtype=np.float32)
vector_y = df['label'].to_numpy(dtype=np.int64)

print(f'Muestras totales: {matriz_x.shape[0]} | Características: {matriz_x.shape[1]}')
print('\nDistribución por clase:')
for (lbl, nom), cant in df.groupby(['label', 'class_name']).size().items():
    pct = (cant / len(df)) * 100.0
    print(f'  [{lbl}] {nom:<12}: {cant:>5} muestras ({pct:.2f}%)')

X_train, X_test, y_train, y_test = train_test_split(
    matriz_x, vector_y, test_size=0.20, stratify=vector_y, random_state=42
)
print(f'\nEntrenamiento: {X_train.shape[0]} muestras | Evaluación: {X_test.shape[0]} muestras')

### 4. Definición de Modelos y Medición de Latencia
Se configuran los dos modelos evaluados para robótica:
1. **Support Vector Machine (RBF)** con `StandardScaler` en Pipeline.
2. **Random Forest Classifier** con 100 estimadores.

In [ ]:
def medir_latencia_ms(modelo, X_eval, repeticiones=300):
    indices = np.random.choice(len(X_eval), size=min(repeticiones, len(X_eval)), replace=False)
    muestras = X_eval[indices]
    _ = modelo.predict(muestras[:1])
    tiempos = []
    for i in range(len(muestras)):
        t0 = time.perf_counter()
        _ = modelo.predict(muestras[i:i+1])
        t1 = time.perf_counter()
        tiempos.append((t1 - t0) * 1000.0)
    return float(np.mean(tiempos))

modelos = {
    'Support Vector Machine (RBF)': Pipeline([
        ('escalador', StandardScaler()),
        ('clasificador', SVC(C=10.0, kernel='rbf', gamma='scale', probability=True, random_state=42))
    ]),
    'Random Forest Classifier': RandomForestClassifier(
        n_estimators=100, max_depth=None, criterion='gini', random_state=42, n_jobs=-1
    )
}
print('Modelos inicializados correctamente.')

### 5. Entrenamiento y Evaluación Empírica

In [ ]:
resultados = {}

for nombre, estimador in modelos.items():
    print(f'--- Entrenando {nombre} ---')
    t_start = time.perf_counter()
    estimador.fit(X_train, y_train)
    t_train = time.perf_counter() - t_start
    
    y_pred = estimador.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1_m = f1_score(y_test, y_pred, average='macro')
    f1_w = f1_score(y_test, y_pred, average='weighted')
    lat_ms = medir_latencia_ms(estimador, X_test)
    rep = classification_report(y_test, y_pred, target_names=nombres_clases, output_dict=True)
    cm = confusion_matrix(y_test, y_pred, normalize='true')
    
    resultados[nombre] = {
        'modelo': estimador,
        'tiempo_s': t_train,
        'acc': acc,
        'f1_macro': f1_m,
        'f1_weighted': f1_w,
        'latencia_ms': lat_ms,
        'reporte': rep,
        'cm': cm,
        'f1_clases': [rep[c]['f1-score'] for c in nombres_clases]
    }
    print(f'Exactitud: {acc*100:.2f}% | F1-Macro: {f1_m*100:.2f}% | Latencia: {lat_ms:.3f} ms/muestra')

df_comparativo = pd.DataFrame([{
    'Modelo': k,
    'Accuracy (%)': f"{v['acc']*100:.2f}%",
    'F1-Macro (%)': f"{v['f1_macro']*100:.2f}%",
    'Latencia (ms)': f"{v['latencia_ms']:.3f} ms",
    'Entrenamiento (s)': f"{v['tiempo_s']:.2f} s"
} for k, v in resultados.items()])
display(df_comparativo)

### 6. Matrices de Confusión Normalizadas (300 DPI)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7), sharey=True)
paletas = ['Blues', 'Greens']

for idx, (nombre, res) in enumerate(resultados.items()):
    sns.heatmap(
        res['cm'] * 100.0, annot=True, fmt='.1f', cmap=paletas[idx], cbar=True, ax=axes[idx],
        xticklabels=nombres_clases, yticklabels=nombres_clases, vmin=0, vmax=100,
        annot_kws={'size': 9, 'weight': 'bold'}
    )
    axes[idx].set_title(f"{nombre}\nAccuracy: {res['acc']*100:.2f}% | F1: {res['f1_macro']*100:.2f}%", fontsize=12, pad=12)
    axes[idx].set_xlabel('Clase Predicha', fontsize=11)
    if idx == 0: axes[idx].set_ylabel('Clase Verdadera', fontsize=11)
    axes[idx].tick_params(axis='x', rotation=35)

plt.suptitle('Matrices de Confusión Normalizadas (%) - LSP', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('matrices_confusion_lsp.png', dpi=300, bbox_inches='tight')
plt.show()

### 7. Comparativa de F1-Score por Clase

In [ ]:
plt.figure(figsize=(13, 6))
indices = np.arange(len(nombres_clases))
ancho = 0.38
colores = ['#1f77b4', '#2ca02c']

for idx, (nombre, res) in enumerate(resultados.items()):
    pos = indices - ancho/2 + idx * ancho
    barras = plt.bar(pos, [v * 100 for v in res['f1_clases']], width=ancho,
                     label=f"{nombre} ({res['f1_macro']*100:.2f}%)", color=colores[idx], edgecolor='black', alpha=0.88)
    for b in barras:
        h = b.get_height()
        plt.annotate(f'{h:.1f}%', xy=(b.get_x() + b.get_width()/2, h), xytext=(0, 3),
                     textcoords='offset points', ha='center', va='bottom', fontsize=8, fontweight='semibold')

plt.title('Comparativa de F1-Score por Clase de Seña (SVM vs Random Forest)', fontsize=13, fontweight='bold', pad=15)
plt.xlabel('Clases (LSP)', fontsize=11); plt.ylabel('F1-Score (%)', fontsize=11)
plt.xticks(indices, nombres_clases, fontsize=10); plt.ylim(0, 110)
plt.grid(axis='y', linestyle='--', alpha=0.5); plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('comparativa_f1_por_clase.png', dpi=300, bbox_inches='tight')
plt.show()

### 8. Empaquetado y Descarga del Modelo Ganador para el Robot NAO

In [ ]:
ganador_nombre = max(resultados.keys(), key=lambda k: resultados[k]['f1_macro'])
ganador = resultados[ganador_nombre]

paquete_produccion = {
    'modelo': ganador['modelo'],
    'nombre_modelo': ganador_nombre,
    'clases': nombres_clases,
    'columnas_caracteristicas': columnas_caracteristicas,
    'metricas': {
        'exactitud_global': ganador['acc'],
        'f1_macro': ganador['f1_macro'],
        'latencia_inferencia_ms': ganador['latencia_ms'],
        'tiempo_entrenamiento_seg': ganador['tiempo_s']
    }
}

joblib.dump(paquete_produccion, 'modelo_lsp_ganador.joblib')
print(f'Modelo ganador exportado: {ganador_nombre}')

# Descarga automática de artefactos para el reporte técnico
for archivo in ['modelo_lsp_ganador.joblib', 'matrices_confusion_lsp.png', 'comparativa_f1_por_clase.png']:
    files.download(archivo)